# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()


# 15. 教職員の権限(申請 → 管理者の承認 → Logto の組織)

**やりたいこと:** 教職員に「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」権限を与える。
**本人が申請し、管理者が承認したときだけ付く**(2026-09-18 に決定。学生と教職員でメールのドメインが同じため、自動では見分けられない)。

| 与える権限 | いまの仕組みでは |
|---|---|
| 地図のパスワードを聞かれない | 公開ページは `$_SESSION['km_map_unlocked']`(パスワード解除)だけを見ている |
| 教職員氏名が見える | `map-access` の `mode`(hidden / password / public)で決まる |
| 「閲覧不可」の地点(`staffOnly`)が見える | Android は `staff:event:access` の権限で判定(`api/app-map.php`)。**公開ページには無い** |

## 0. 決めたこと(2026-09-18)

| 決めごと | 内容 |
|---|---|
| 誰を教職員とみなすか | **本人の申請を、管理者が管理画面で承認した人**(`km_staff_requests`)。`customData.isteacher` と Logto の JIT は**使わない**(自己申告だけで付けない・学生と同じドメイン) |
| `customData` を誰が書けるか | 使わないが、**本人が書けない状態(`Off` か `ReadOnly`)を保つ**。本番は 2026-09-18 に `ReadOnly` |
| 権限の載せ方 | **組織 + 組織ロール**(利用者の希望)。組織 `Kosen_Member`(`0kpbyqtgrkcc`)/ ロール `Kosen_Member` / API 権限 `staff:normal:access` |
| 付け方 | **承認した瞬間に**組織へ入れる。取り消したら外す(Management API)。記録は管理操作ログにも残す |
| 通知 | 申請が来たら管理者へメール(既存の送信サーバー)。**Teams への送信は後の段**(自動送信の口を作る方針) |
| イベント運営との関係 | **別の権限。** イベント運営は `staff:event:access`、教職員は `staff:normal:access`。両方持つ人は両方効く |
| `hidden` のとき | **管理画面に切り替えを足す。** 既定は「hidden なら教職員にも隠す」 |
| SMS | **入れない。** 確認はメールの確認コード(自前の送信サーバー。無料) |

> **判定はトークンの中だけで行う。** 申請の表や Management API を毎回の判定に使うと、遅いうえに
> Logto が落ちたときに地図まで止まる。**申請の表は「誰を承認したか」の記録**、**実際の判定は組織ロール(トークンの権限)**。

## 1. Logto の仕様で、先に知っておくこと

公式ドキュメントで確かめた(2026-09-18):

- **JIT(ドメインでの自動付与)は使わない。** ドメイン単位でしか設定できず、学生も同じドメインだから
- **組織への追加はこちらで行う。** `customData.isteacher` を見て、Management API で組織に入れる(サインインのとき)
- 本人が自分で変えられる項目(Account API)は `name` / `profile` / `email` / `phone` / `password` / `passkey`。**`customData` は Off**
- **組織ロールは「組織トークン」に載る。** API は `aud = https://ito4.jp/api` に加えて `organization_id` を持つトークンを受け取れるようにする必要がある
  - Android: `getAccessToken("https://ito4.jp/api", organizationId)`
  - 公開ページ(PHP): ID トークンの `organization_roles`(`org_xxx:role` の形)を見る。`urn:logto:scope:organization_roles` を要求する

## 2. 段取り(1 つずつ。段ごとに検証機 → 本番)

| 段 | 何をする | 誰が | 状態 |
|---|---|---|---|
| 1 | Logto Console で**組織**(`Kosen_Member` / `0kpbyqtgrkcc`)と**組織ロール**を作り、ロールに API 権限 `staff:normal:access` を割り当てる | 自分で(Console) | **済(2026-09-18)** |
| A | 組織への出し入れ(`lib/staff-org.php`)と申請の表(`km_staff_requests`)、設定値(`KM_LOGTO_ORG_ID` ほか) | 任せる | **検証機で済(2026-09-18)** |
| B | 申請(`account.php` の「教職員の権限」欄)・承認/却下/取り消しの管理画面(`admin/staff-requests.php`)・申請のメール通知 | 任せる | **検証機で済(2026-09-18)**(申請→通知→承認→取り消しを利用者が画面で確認) |
| C | API(`logto_guard.php`)が**組織トークン**を受け付ける(組織 ID を照合・採るのは教職員の権限だけ・組織外の教職員の権限は捨てる)。公開ページは ID トークンの `organization_roles` を見る | 任せる | **検証機で済(2026-09-18)**(公開ページは利用者が「教職員として地図の情報を見られます」を確認 = `organization_roles` は `組織ID:ロール名`。API の判定は check.php で偽のトークン内容から確認。本物のトークンは段 E で) |
| D | 地図の扱い: 教職員は**パスワード無しで地図と氏名**(`password` のとき)。`hidden` は既定で隠し、管理画面の「常に隠すでも教職員には見せる」で出す。アプリは教職員にも**閲覧不可の地点**を配る。公開ページの印は ID トークンの期限まで・停止中は立てない・サインアウトで外す | 任せる | **検証機で済(2026-09-18)**(利用者が 錠の素通り・hidden の切り替え・サインアウトで戻る を確認) |
| G | **教職員が自分の地点を編集できる**(管理者が地点の UUID を教職員に割り当て、その地点だけ編集可)。**地点の文字の項目(名前・部屋番号など・教職員氏名・メモ)はすべて編集可・位置/種類/経路は不可・在室表示は作らない(Teams で見られる)・反映の前に管理者の確認を挟む**(2026-09-18 決定)。`lib/staff-nodes.php`・`account.php` の「担当の地点」・管理画面「教職員の地点」(`admin/staff-nodes.php`)・地図編集に「地点の ID」。プライバシーポリシー 5 節に追記 | 任せる | **実装済み・check.php 1204 件通過・実機(DB)での確認待ち**(2026-09-18。検証機が止まっていたため未配備) |
| E | Android: 地図の取得だけ組織付きのトークン(`getMapAccessTokenOrNull`。取れなければ通常のトークン)。サインインに `urn:logto:scope:organizations`。役割に **教職員(TEACHER)** を足す(氏名・閲覧不可の地点は見える・通行止めは通れない)。教職員かは組織付きのトークンで `logto_me.php` に聞き直す | 任せる(署名と配布は自分で) | **本番で実機確認済み(2026-09-18)**(端末の hosts を書けないため検証機ではなく本番で。本番の組織に残っていた JIT を先に外した)(検証機向けは `-Pkosenmap.staffOrgId` と `-Pkosenmap.apiResource` を付けて作る。ノート 13) |
| F | Teams への自動送信の口(申請の通知・承認の知らせ) | 任せる | 未 |

**段 A の検証機の結果(2026-09-18):** 試験用の組織 `vpguys770lr3` を作り、実際の利用者で
付ける → 2 回目も成功 → 持っている → 外す → 2 回目も成功 → 持っていない、を確認。
申請 → 二重の申請は断る → 承認で権限が付く → 二重の承認は断る → 取り消しで外れる → 却下では付かない、も確認。
**組織の側から役割を読むと、居ない人は 422 になってエラーログが増えた**ので、利用者の側(`/users/{id}/organizations`)から読むようにした。

**段 1 と A は済んでいる。** 次は段 B(画面)。本番の `.env` に組織 ID を入れるのは、段 B を配備するとき。


## 3. 段 1 の結果(2026-09-18 に本番で実測)

| 何 | 値 |
|---|---|
| 組織 | `Kosen_Member` —— ID **`0kpbyqtgrkcc`**(説明「高専メンバー」) |
| 組織ロール | `Kosen_Member`(種別 User) |
| ロールに付いている API 権限 | **`staff:normal:access`**(`KosenMap PHP API` = `https://ito4.jp/api`) |
| JIT のメールドメイン | **設定しない**(学生と同じドメインのため)。**★ 2026-09-18 の実測では `ichinoseki.kosen-ac.jp`・既定ロール `Kosen_Member` が残っていた(人数 0)。Console で外すまで本番の `.env` に `KM_LOGTO_ORG_ID` を入れない**(入れると、学生がサインアップしただけで教職員になる) |
| Account API の `customData` | **`Off`**(本人は書き換えられない = 自己申告で教職員になれない) |

下のセルは、この状態を**読むだけ**で確かめ直す。`customData` が `Off` でなくなっていたら赤字で出す。

🟢 **読むだけ** —— 何も変えません。


In [ ]:
%%host --timeout 180
# 組織・組織ロール・Account API の設定を読むだけ(書き換えない)。参照用の資格情報を使う
docker compose exec -T -u www-data web php -r '
require "/var/www/html/lib/logto-management.php";
$orgs = km_logto_management_get("organizations", [], "readonly");
foreach ($orgs as $o) {
    printf("組織      : %s  %s\n", (string) ($o["id"] ?? "?"), (string) ($o["name"] ?? "?"));
}
foreach (km_logto_management_get("organization-roles", [], "readonly") as $r) {
    $scopes = array_map(static fn ($s) => (string) ($s["name"] ?? "?"), (array) ($r["resourceScopes"] ?? []));
    printf("組織ロール: %s  権限 %s\n", (string) ($r["name"] ?? "?"), $scopes === [] ? "(無し)" : implode(" ", $scopes));
}
$ac = km_logto_management_get("account-center", [], "readonly");
$fields = (array) ($ac["fields"] ?? []);
printf("Account API: enabled=%s / customData=%s%s\n",
    ($ac["enabled"] ?? false) ? "true" : "false",
    (string) ($fields["customData"] ?? "?"),
    ($fields["customData"] ?? "Off") === "Off" ? "  OK" : "  ★ Off に戻すこと(自己申告で教職員になれる)");
printf("退会の案内先: %s\n", (string) ($ac["deleteAccountUrl"] ?? "(無し)"));
' </dev/null


## 4. 実装の形

```
本人(サインイン済み)
  account.php → 教職員の申請ページ(ひとこと: 氏名・所属)
        │ km_staff_requests に pending で入る / 管理者へメール(のちに Teams)
        ▼
管理者(管理画面)
  申請の一覧 → 承認 / 却下 / (承認済みを)取り消し
        │ 承認 = Logto の組織 0kpbyqtgrkcc に入れて組織ロール Kosen_Member を付ける(lib/staff-org.php)
        │ 取り消し = 組織から外す
        ▼
組織ロール Kosen_Member → 権限 staff:normal:access
        ├─ Android: getAccessToken("https://ito4.jp/api", "0kpbyqtgrkcc")
        │            → aud=https://ito4.jp/api・organization_id=0kpbyqtgrkcc・scope に staff:normal:access
        │            → api/app-map.php が「閲覧不可の地点」と「教職員氏名」を出す
        └─ 公開ページ: ID トークンの organization_roles → 錠を通す・教職員氏名を出す(hidden は §段 D の切り替え次第)
```

**守りの要点:**

- **判定はトークンの中だけで完結させる。** 申請の表や Management API を判定経路に入れない
- **`organization_id` を必ず照合する。** `aud` だけ見ると、別の組織のトークンでも通る
- **承認は Logto を先に変えてから表を書く。** 逆だと「承認済みなのに権限が無い」が残る。同時に 2 人が押しても 1 回しか通らない
- **取り消しても、発行済みのトークンは期限(最長 1 時間)まで効く。** 急ぐときは Logto で本人のセッションも切る
- **退会したら申請の記録も消す**(`lib/account-delete.php` の対象に入れた)
- **期限の切れた ID トークンは信じない**(2026-09-25、診断 16 の W-50・W-51)。公開ページと `account.php` は、期限の近い ID トークンを
  refresh_token で取り直してから `organization_roles` を見る(`KmLogtoClient::kmFreshIdTokenClaims`)。取り直せなければ教職員の特典を出さない

**運用の約束(2026-09-25):** 教職員の権限を外すときは、**管理画面の「取り消し」を使う。Logto Console で組織から直接外さない。**
Console で外すと申請の表が `approved` のまま残り、地点の変更の提案を承認するときの見直し(`km_staff_node_edit_decide`)をすり抜ける。

## 5. ついでに見つかった古い設定

**退会の案内先が旧ドメインのまま**だった(実測 `https://ito8795.com/account.php`)。旧ドメインは手放す予定なので、
`https://ito4.jp/account.php` に直す必要がある。**Console の「アカウント設定」から直せる**(こちらからも Management API で直せる)。

🟡 **Logto の設定が変わる** —— 実行前に `yes` の入力を求めます。


In [ ]:
%%host --confirm "Logto の退会の案内先を https://ito4.jp/account.php に直します" --timeout 120
docker compose exec -T -u www-data web php -r '
require "/var/www/html/lib/logto-management.php";
$before = km_logto_management_get("account-center", [], "readonly");
printf("いま: %s\n", (string) ($before["deleteAccountUrl"] ?? "(無し)"));
' </dev/null
echo "(書き換えは段 2 以降で lib に口を足してから。ここでは今の値だけ出す)"
